# cAPTure: graph-training runner binding

This notebook binds the one-seed graph-training runner to the approved immutable selection materialization and complete-fold graph collection. It freezes the twelve model/fold jobs, exact architecture, optimization, weighting, early-stopping, final-refit, persistence, and pilot-decision contracts.

It does not instantiate a model, run inference, perform optimization, read an outer-validation score, or access Test1, Test2, `train_pub_exf`, or `train_user_prop`. A separate manual authorization, bound to the generated report and job-plan hashes, is required before any backward pass.

## 1. Mount Drive and prepare the environment

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_SELECTION_PARENT = Path("/content/capture_graph_selection_inputs")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_training_binding import (
    run_capture_graph_training_binding,
    save_graph_training_authorization,
    stage_capture_selection_materialization,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Training-runner binding environment is ready.")

Mounted at /content/drive
Repository commit: 11ecd7a45a3e1ca5408a6c80ac4472fb4f32e947
Training-runner binding environment is ready.


## 2. Bind the approved immutable inputs

In [2]:
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
TRAINING_PREFLIGHT_RUN_ID = (
    "20260929T131750_353987Z_graph_training_preflight_v2"
)
SELECTION_MATERIALIZATION_RUN_ID = (
    "20260929T134737_484622Z_graph_selection_materialization"
)
SELECTION_MATERIALIZATION_MANIFEST_SHA256 = (
    "846d21671d6c657903427fefe665e8d1919c74859703a068a4940d02d8c09633"
)

GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
TRAINING_PREFLIGHT_DIR = (
    DRIVE_ROOT / "graph_training_preflight_v2_runs" / TRAINING_PREFLIGHT_RUN_ID
)
SELECTION_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_selection_materialization_runs"
    / SELECTION_MATERIALIZATION_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
PREFLIGHT_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_preflight_v2.yaml"
)
SELECTION_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_selection_materialization_v1.yaml"
)
TRAINING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_v1.yaml"
)

for path in (
    GRAPH_MATERIALIZATION_DIR, TRAINING_PREFLIGHT_DIR,
    SELECTION_MATERIALIZATION_DIR, GRAPH_INPUT_CONTRACT_PATH,
    PREFLIGHT_CONFIG_PATH, SELECTION_CONFIG_PATH, TRAINING_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")

print("Approved selection materialization:", SELECTION_MATERIALIZATION_DIR)
print("Frozen training contract:", TRAINING_CONFIG_PATH)

Approved selection materialization: /content/drive/MyDrive/capture_gate0/graph_selection_materialization_runs/20260929T134737_484622Z_graph_selection_materialization
Frozen training contract: /content/temporalgnn-nids/configs/capture_graph_training_v1.yaml


## 3. Stage and verify both graph collections

The complete-fold collection supplies final-refit and outer-validation tensors. The selection collection supplies only the fold-training scenarios transformed by the inner-training-only preprocessors. Both collections are copied to local Colab storage before repeated access.

In [3]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
selection_root, selection_manifest = stage_capture_selection_materialization(
    source_root=SELECTION_MATERIALIZATION_DIR,
    local_parent=LOCAL_SELECTION_PARENT,
    collection=collection,
    config_path=SELECTION_CONFIG_PATH,
    expected_run_id=SELECTION_MATERIALIZATION_RUN_ID,
    expected_manifest_sha256=SELECTION_MATERIALIZATION_MANIFEST_SHA256,
)
print("Verified complete-fold graphs:", collection.root)
print("Verified selection graphs:", selection_root)

Staged and verified graph materialization: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
Staged and verified selection materialization: /content/capture_graph_selection_inputs/20260929T134737_484622Z_graph_selection_materialization
Verified complete-fold graphs: /content/capture_graph_stage2_inputs_v2/20260928T175821_521784Z_graph_materialization_full_dev
Verified selection graphs: /content/capture_graph_selection_inputs/20260929T134737_484622Z_graph_selection_materialization


## 4. Create or reuse the immutable runner binding

This step constructs metadata and verifies data ranges and weights. It does not instantiate or train a model.

In [4]:
BINDING_RUN_ID = None  # Paste an existing completed ID to reuse it.
if BINDING_RUN_ID is None:
    BINDING_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_training_binding"
    )
BINDING_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_training_binding_runs" / BINDING_RUN_ID
)
print("Training-runner binding ID:", BINDING_RUN_ID)
print("Training-runner binding output:", BINDING_OUTPUT_DIR)

binding_report = run_capture_graph_training_binding(
    collection=collection,
    selection_root=selection_root,
    preflight_dir=TRAINING_PREFLIGHT_DIR,
    training_config_path=TRAINING_CONFIG_PATH,
    preflight_config_path=PREFLIGHT_CONFIG_PATH,
    selection_config_path=SELECTION_CONFIG_PATH,
    output_dir=BINDING_OUTPUT_DIR,
)
print(json.dumps(binding_report, indent=2))

Training-runner binding ID: 20260929T144234_934044Z_graph_training_binding
Training-runner binding output: /content/drive/MyDrive/capture_gate0/graph_training_binding_runs/20260929T144234_934044Z_graph_training_binding
{
  "report_version": 1,
  "status": "review_required",
  "runner_binding_run_id": "20260929T144234_934044Z_graph_training_binding",
  "run_config_sha256": "bf8dc7d06d04a6a16a5ba01fe9f6f97950060584c85d41b42ddda770a234b80e",
  "training_contract_sha256": "e0ea649e3a98a762a9038e8b000d8d59ed6f880926b9cb61673810dd12103c93",
  "training_job_plan_sha256": "c3d64778323832919ad1f778d42a50a4d626ae80ee6c074bddc0e45cff5407c8",
  "selection_materialization_run_id": "20260929T134737_484622Z_graph_selection_materialization",
  "selection_materialization_manifest_sha256": "846d21671d6c657903427fefe665e8d1919c74859703a068a4940d02d8c09633",
  "selection_materialization_decision_sha256": "55fbeefdb87ca2ada129116a1bdc80e173b5923abd8163324b9ac828bae10f92",
  "selection_materialization_appro

## 5. Review the exact jobs, slices, and weights

In [5]:
job_plan = json.loads(
    (BINDING_OUTPUT_DIR / "training_job_plan.json").read_text(encoding="utf-8")
)
display(pd.DataFrame(job_plan["jobs"]))

slice_rows = []
weight_rows = []
for fold, fold_plan in job_plan["fold_plans"].items():
    for scenario, sequence in fold_plan["selection_sequences"].items():
        slice_rows.append({"fold": fold, "scenario": scenario, **sequence})
        weight_rows.append({
            "fold": fold,
            "phase": "selection",
            "scenario": scenario,
            **fold_plan["selection_scenario_class_weights"][scenario],
        })
        weight_rows.append({
            "fold": fold,
            "phase": "complete_refit",
            "scenario": scenario,
            **fold_plan["complete_refit_scenario_class_weights"][scenario],
        })
display(pd.DataFrame(slice_rows))
display(pd.DataFrame(weight_rows))
display(pd.DataFrame.from_dict(job_plan["model_specifications"], orient="index"))
print("Binding status:", binding_report["status"])
print("Jobs frozen:", binding_report["jobs"])
print("Training performed:", binding_report["training_performed"])
print("Held-out scenarios accessed:", binding_report["held_out_scenarios_accessed"])

,job_id,model,fold,selection_training_scenarios,outer_validation_scenarios,status
0,edge_mlp__fold_A,edge_mlp,A,"[train_empty_conn, train_qos_mid]","[train_dollar_char, train_slash_char, train_su...",awaiting_manual_training_authorization
1,edge_mlp__fold_B,edge_mlp,B,"[train_dollar_char, train_slash_char, train_su...","[train_empty_conn, train_qos_mid]",awaiting_manual_training_authorization
2,edge_gru__fold_A,edge_gru,A,"[train_empty_conn, train_qos_mid]","[train_dollar_char, train_slash_char, train_su...",awaiting_manual_training_authorization
3,edge_gru__fold_B,edge_gru,B,"[train_dollar_char, train_slash_char, train_su...","[train_empty_conn, train_qos_mid]",awaiting_manual_training_authorization
4,static_gnn__fold_A,static_gnn,A,"[train_empty_conn, train_qos_mid]","[train_dollar_char, train_slash_char, train_su...",awaiting_manual_training_authorization
5,static_gnn__fold_B,static_gnn,B,"[train_dollar_char, train_slash_char, train_su...","[train_empty_conn, train_qos_mid]",awaiting_manual_training_authorization
6,st_gnn__fold_A,st_gnn,A,"[train_empty_conn, train_qos_mid]","[train_dollar_char, train_slash_char, train_su...",awaiting_manual_training_authorization
7,st_gnn__fold_B,st_gnn,B,"[train_dollar_char, train_slash_char, train_su...","[train_empty_conn, train_qos_mid]",awaiting_manual_training_authorization
8,st_gnn_without_gat__fold_A,st_gnn_without_gat,A,"[train_empty_conn, train_qos_mid]","[train_dollar_char, train_slash_char, train_su...",awaiting_manual_training_authorization
9,st_gnn_without_gat__fold_B,st_gnn_without_gat,B,"[train_dollar_char, train_slash_char, train_su...","[train_empty_conn, train_qos_mid]",awaiting_manual_training_authorization


,fold,scenario,scenario_report,inner_train_graph_range,inner_validation_graph_range,excluded_post_validation_graph_range,validation_window_index_range,inner_train_edges,inner_validation_edges,excluded_post_validation_edges
0,A,train_empty_conn,fold_A/train_empty_conn/scenario_fold_report.json,"[0, 2529]","[2529, 4401]","[4401, 9360]","[2529, 4401]",593753,168690,413336
1,A,train_qos_mid,fold_A/train_qos_mid/scenario_fold_report.json,"[0, 6716]","[6716, 8588]","[8588, 9360]","[6716, 8588]",1226928,215905,56884
2,B,train_dollar_char,fold_B/train_dollar_char/scenario_fold_report....,"[0, 7451]","[7451, 9083]","[9083, 9116]","[7452, 9324]",2759427,122977,151
3,B,train_slash_char,fold_B/train_slash_char/scenario_fold_report.json,"[0, 7478]","[7478, 9109]","[9109, 9116]","[7480, 9352]",6074640,350862,14
4,B,train_sub_exf,fold_B/train_sub_exf/scenario_fold_report.json,"[0, 7459]","[7459, 9090]","[9090, 9115]","[7461, 9333]",2140188,85513,106


,fold,phase,scenario,normal,attack
0,A,selection,train_empty_conn,2.193539,1.178440
1,A,complete_refit,train_empty_conn,0.895646,1.559245
2,A,selection,train_qos_mid,0.836744,0.666477
3,A,complete_refit,train_qos_mid,0.895646,0.888384
4,B,selection,train_dollar_char,1.125788,1.611846
5,B,complete_refit,train_dollar_char,1.127454,1.632465
6,B,selection,train_slash_char,1.121899,0.411545
7,B,complete_refit,train_slash_char,1.127454,0.407225
8,B,selection,train_sub_exf,1.124688,3.558989
9,B,complete_refit,train_sub_exf,1.127454,3.691049


,class,temporal,temporal_memory_policy,parameters
edge_mlp,SimpleMLP,False,none,"{'edge_dim': 103, 'hidden_dim': 64, 'dropout':..."
edge_gru,EdgeGRU_Baseline_NoX,True,exponential_decay,"{'edge_dim': 103, 'hidden_dim': 64, 'dropout':..."
static_gnn,StaticGNN_Identity,False,none,"{'edge_dim': 103, 'hidden_dim': 64, 'dropout':..."
st_gnn,ST_GNN_Identity,True,exponential_decay,"{'edge_dim': 103, 'hidden_dim': 64, 'dropout':..."
st_gnn_without_gat,ST_GNN_Identity,True,exponential_decay,"{'edge_dim': 103, 'hidden_dim': 64, 'dropout':..."
st_gnn_without_direct_edge_attr,ST_GNN_Identity,True,exponential_decay,"{'edge_dim': 103, 'hidden_dim': 64, 'dropout':..."


Binding status: review_required
Jobs frozen: 12
Training performed: False
Held-out scenarios accessed: False


## 6. Optional manual authorization

Review the report, the twelve jobs, graph ranges, separate selection/refit weights, and frozen model parameters first. Leave `AUTHORIZE_MODEL_TRAINING` as `None` until the binding is accepted. This decision authorizes only development-fold graph optimization; it never authorizes Test1 or Test2.

In [6]:
AUTHORIZE_MODEL_TRAINING = True
TRAINING_AUTHORIZATION_NOTE = (
    "Approved for the one-seed development graph pilot. The runner is bound "
    "to the approved selection materialization and complete-fold graph "
    "collection. The twelve model/fold jobs, inner-validation slices, "
    "scenario/class weights, architecture, optimizer, temporal-memory policy, "
    "early-stopping rule, complete-data refit, and OOF output contract are "
    "frozen. This authorization does not permit held-out-scenario access."
)

if AUTHORIZE_MODEL_TRAINING is None:
    print("No training authorization was written. Review the binding first.")
else:
    authorization = save_graph_training_authorization(
        BINDING_OUTPUT_DIR,
        approved=bool(AUTHORIZE_MODEL_TRAINING),
        review_note=TRAINING_AUTHORIZATION_NOTE,
    )
    print("Saved training authorization:", json.dumps(authorization, indent=2))

Saved training authorization: {
  "approved": true,
  "review_note": "Approved for the one-seed development graph pilot. The runner is bound to the approved selection materialization and complete-fold graph collection. The twelve model/fold jobs, inner-validation slices, scenario/class weights, architecture, optimizer, temporal-memory policy, early-stopping rule, complete-data refit, and OOF output contract are frozen. This authorization does not permit held-out-scenario access.",
  "training_runner_binding_report_sha256": "710593fe1a10cbe2342fe6a4dc52c3fa971eeff90bdae6edacb6f3d333457995",
  "training_job_plan_sha256": "c3d64778323832919ad1f778d42a50a4d626ae80ee6c074bddc0e45cff5407c8",
  "selection_materialization_manifest_sha256": "846d21671d6c657903427fefe665e8d1919c74859703a068a4940d02d8c09633",
  "model_training_authorized": true,
  "held_out_scenarios_authorized": false,
  "next_action": "implement_and_run_one_resumable_model_fold_job"
}
